# The Green Grid — Classificacao de Doencas Vegetais

## Arquitetura de Paralelismo (3 niveis)

| Nivel | Tecnologia | Onde atua |
|-------|-----------|----------|
| **1 — CUDA** | PyTorch / GPU cores | Dentro de cada GPU: operacoes matriciais em paralelo nos cores CUDA |
| **2 — CPU** | DataLoader `num_workers` | Dentro de cada no: pre-processamento paralelo de imagens nos nucleos fisicos |
| **3 — Distribuido** | Ray | Entre maquinas: cada no do cluster treina um modelo simultaneamente |

## Como executar
- **No unico (teste local):** execute as celulas normalmente. Ray sobe um cluster local automaticamente.
- **Cluster (2 PCs na mesma rede):**
  1. Desktop (head node): execute normalmente
  2. Notebook PC (worker): `ray start --address='IP_DO_DESKTOP:6379'` no terminal
  3. Re-execute a celula 3 para ver os recursos do cluster
  4. Execute as demais celulas — Ray distribui os 3 modelos automaticamente

## Logica de checkpoint
- Cada epoca salva estado completo: se cair no epoch 7, re-execute e continua do epoch 8.
- Treino concluido salva `_DONE.pt`: re-executar pula automaticamente.


In [1]:
# === 1. IMPORTS ===
import os
import time
from pathlib import Path
from collections import Counter

import numpy as np
import ray
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score, confusion_matrix
)
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

KeyboardInterrupt: 

In [ ]:
# === 2. CAMINHOS E HIPERPARAMETROS ===
BASE_DIR   = Path('.').resolve()
DATA_DIR   = BASE_DIR / 'PlantVillage-completo'
SPLITS_DIR = DATA_DIR / 'splits'
RAW_DIR    = DATA_DIR / 'raw'
CKPT_DIR   = BASE_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 128
NUM_EPOCHS  = 10
LR          = 1e-3
NUM_WORKERS = 0   # Nivel 2: nucleos CPU para pre-processamento paralelo de imagens
SEED        = 42
VERSIONS    = ['color', 'grayscale', 'segmented']

CONFIG = {
    'base_dir':    str(BASE_DIR),
    'data_dir':    str(DATA_DIR),
    'splits_dir':  str(SPLITS_DIR),
    'ckpt_dir':    str(CKPT_DIR),
    'img_size':    IMG_SIZE,
    'batch_size':  BATCH_SIZE,
    'num_epochs':  NUM_EPOCHS,
    'lr':          LR,
    'num_workers': NUM_WORKERS,
    'seed':        SEED,
    'class_to_idx': {name: i for i, name in enumerate(sorted([d.name for d in (Path('.').resolve() / 'PlantVillage-completo' / 'raw' / 'color').iterdir() if d.is_dir()]))},
}

print(f'Base : {BASE_DIR}')
print(f'Dados: {DATA_DIR}')
print(f'Ckpts: {CKPT_DIR}')
print(f'IMG_SIZE={IMG_SIZE}  BATCH_SIZE={BATCH_SIZE}  NUM_EPOCHS={NUM_EPOCHS}  NUM_WORKERS={NUM_WORKERS}')

Base : C:\Users\yluan\Documents\luan\TI6
Dados: C:\Users\yluan\Documents\luan\TI6\PlantVillage-completo
Ckpts: C:\Users\yluan\Documents\luan\TI6\checkpoints
IMG_SIZE=224  BATCH_SIZE=128  NUM_EPOCHS=10  NUM_WORKERS=0


In [ ]:
# === 3. RAY INIT + DEVICE ===
# Nivel 3: inicializa o cluster Ray
#   - No unico: sobe cluster local automaticamente
#   - Com worker PCs conectados via 'ray start --address=...', aparecem aqui

import torch
import numpy as np

torch.manual_seed(SEED)
np.random.seed(SEED)

if not ray.is_initialized():
    ray.init(ignore_reinit_error=True, log_to_driver=True)

resources = ray.cluster_resources()
nodes = ray.nodes()
print(f'Cluster Ray — {len(nodes)} no(s):')
for n in nodes:
    status = 'ATIVO' if n['Alive'] else 'INATIVO'
    addr   = n['NodeManagerAddress']
    cpus   = n['Resources'].get('CPU', 0)
    gpus   = n['Resources'].get('GPU', 0)
    print(f'  [{status}] {addr}  CPUs={cpus:.0f}  GPUs={gpus:.0f}')
print(f'Total — CPUs: {resources.get("CPU",0):.0f}  GPUs: {resources.get("GPU",0):.0f}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'\nHead node GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
else:
    print('Head node: CPU')

2026-03-31 14:22:29,994	INFO worker.py:2013 -- Started a local Ray instance.


Cluster Ray — 1 no(s):
  [ATIVO] 127.0.0.1  CPUs=16  GPUs=1
Total — CPUs: 16  GPUs: 1

Head node GPU: NVIDIA GeForce RTX 3050  (8.6 GB)


In [ ]:
# === 4. CLASSES DO DATASET ===
CLASS_NAMES = sorted([d.name for d in (RAW_DIR / 'color').iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print(f'{NUM_CLASSES} classes encontradas')
for i, name in enumerate(CLASS_NAMES[:5]):
    print(f'  {i:02d}  {name}')
print(f'  ... (+{NUM_CLASSES - 5} mais)')

38 classes encontradas
  00  Apple___Apple_scab
  01  Apple___Black_rot
  02  Apple___Cedar_apple_rust
  03  Apple___healthy
  04  Blueberry___healthy
  ... (+33 mais)


In [ ]:
# === 5. CLASSE DO DATASET ===
# O label e derivado do nome da pasta (classe) no caminho da imagem.
# Formato do split: raw/{version}/{class_name}/{filename}
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

class PlantVillageDataset(Dataset):
    def __init__(self, split_file, data_dir, class_to_idx, transform=None):
        self.data_dir     = Path(data_dir)
        self.transform    = transform
        self.class_to_idx = class_to_idx
        self.samples      = []
        with open(split_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # extrai classe do caminho: raw/version/CLASS_NAME/arquivo
                parts = line.split("/")
                class_name = parts[2]
                label = class_to_idx[class_name]
                self.samples.append((line, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        rel_path, label = self.samples[idx]
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


for ver in VERSIONS:
    n_train = sum(1 for _ in open(SPLITS_DIR / f"{ver}_train.txt"))
    n_test  = sum(1 for _ in open(SPLITS_DIR / f"{ver}_test.txt"))
    print(f"  {ver:12s} train: {n_train:,} | test: {n_test:,}")

  color        train: 43,596 | test: 10,709
  grayscale    train: 43,203 | test: 11,102
  segmented    train: 42,984 | test: 11,322


In [ ]:
# === 6. EDA — DISTRIBUICAO POR CULTURA ===
EDA_DIST = BASE_DIR / 'eda_distribuicao.png'
if EDA_DIST.exists():
    print('eda_distribuicao.png ja existe — pulando')
else:
    culture_counts = Counter()
    with open(SPLITS_DIR / 'color_train.txt') as f:
        for line in f:
            parts = line.strip().split('/')
            if len(parts) >= 3:
                culture_counts[parts[2].split('___')[0]] += 1
    cultures = sorted(culture_counts.keys())
    counts   = [culture_counts[c] for c in cultures]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(cultures, counts, color='steelblue')
    ax.set_title('Distribuicao por Cultura (color train set)')
    ax.set_xlabel('Cultura'); ax.set_ylabel('Numero de imagens')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig(EDA_DIST, dpi=120); plt.close()
    print(f'Salvo: {EDA_DIST}')

eda_distribuicao.png ja existe — pulando


In [ ]:
# === 7. EDA — COMPARACAO DE VERSOES ===
EDA_VER = BASE_DIR / 'eda_versoes.png'
if EDA_VER.exists():
    print('eda_versoes.png ja existe — pulando')
else:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, ver in enumerate(VERSIONS):
        with open(SPLITS_DIR / f'{ver}_train.txt') as f:
            first_line = f.readline().strip().split()[0]
        img = Image.open(DATA_DIR / first_line).convert('RGB')
        axes[i].imshow(img); axes[i].set_title(ver); axes[i].axis('off')
    plt.suptitle('Versoes do PlantVillage'); plt.tight_layout()
    plt.savefig(EDA_VER, dpi=120); plt.close()
    print(f'Salvo: {EDA_VER}')

eda_versoes.png ja existe — pulando


In [ ]:
# === 8. FILTRO ExG (Excess Green) — demonstracao ===
EXG_PATH = BASE_DIR / 'exg_demo.png'
if EXG_PATH.exists():
    print('exg_demo.png ja existe — pulando')
else:
    with open(SPLITS_DIR / 'color_train.txt') as f:
        first_line = f.readline().strip().split()[0]
    img = np.array(Image.open(DATA_DIR / first_line).convert('RGB'))
    r, g, b = img[:,:,0]/255., img[:,:,1]/255., img[:,:,2]/255.
    exg      = 2*g - r - b
    exg_norm = (exg - exg.min()) / (exg.max() - exg.min() + 1e-8)
    mask      = exg > 0.1
    segmented = img.copy(); segmented[~mask] = 0
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img);       axes[0].set_title('Original');       axes[0].axis('off')
    axes[1].imshow(exg_norm, cmap='RdYlGn'); axes[1].set_title('ExG (mapa)'); axes[1].axis('off')
    axes[2].imshow(segmented); axes[2].set_title('Segmentado (ExG)'); axes[2].axis('off')
    plt.suptitle('Filtro Excess Green (ExG)'); plt.tight_layout()
    plt.savefig(EXG_PATH, dpi=120); plt.close()
    print(f'Salvo: {EXG_PATH}')

exg_demo.png ja existe — pulando


## Arquitetura de Paralelismo — Diagrama

```
Ray (Nivel 3 — Distribuido entre nos do cluster)
  |
  +-- No 1 (Desktop, RTX 3050 8GB): treina modelo COLOR
  |     |
  |     +-- DataLoader num_workers=4 (Nivel 2 — CPU paralela)
  |     |     4 processos carregam e pre-processam imagens simultaneamente
  |     |     enquanto a GPU processa o batch atual (pipeline de dados)
  |     |
  |     +-- GPU / CUDA cores (Nivel 1 — CUDA paralela)
  |           milhares de cores CUDA processam o batch matricialmente em paralelo
  |
  +-- No 2 (Notebook PC, RTX 3050 6GB): treina modelo GRAYSCALE
  |     +-- DataLoader num_workers (Nivel 2)
  |     +-- GPU / CUDA cores (Nivel 1)
  |
  +-- No 1 ou 2: treina modelo SEGMENTED
        (Ray aloca no no com GPU disponivel)
```

**Fase Map:** cada no processa sua versao do dataset de forma independente.
**Fase Reduce:** meta-classificador combina as predicoes dos 3 modelos (ensemble).


In [ ]:
# === 10. MODELTRAINER — RAY ACTOR ===
#
# @ray.remote(num_gpus=0.34):
#   Nivel 3 — cada actor e um no distribuido do cluster Ray
#   num_gpus=0.34 -> single machine: 3 actors compartilham 1 GPU
#                 -> cluster 2 PCs: Ray aloca 1 actor por GPU automaticamente

@ray.remote(num_gpus=0.34)
class ModelTrainer:

    def __init__(self, version, config):
        import torch
        from torch.utils.data import DataLoader, Dataset
        from torchvision import transforms
        from pathlib import Path
        from PIL import Image

        class _DS(Dataset):
            def __init__(self, split_file, data_dir, class_to_idx, transform=None):
                self.data_dir  = Path(data_dir)
                self.transform = transform
                self.samples   = []
                with open(split_file) as f:
                    for line in f:
                        line = line.strip()
                        if not line: continue
                        parts = line.split("/")
                        class_name = parts[2]
                        label = class_to_idx[class_name]
                        self.samples.append((line, label))
            def __len__(self): return len(self.samples)
            def __getitem__(self, idx):
                p, lbl = self.samples[idx]
                img = Image.open(self.data_dir / p).convert('RGB')
                return self.transform(img) if self.transform else img, lbl

        self.version = version
        self.config  = config
        self.device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        data_dir   = Path(config['data_dir'])
        splits_dir = Path(config['splits_dir'])
        img_size   = config['img_size']
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]

        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        eval_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

        train_ds      = _DS(splits_dir / f'{version}_train.txt', data_dir, config["class_to_idx"], transform=train_tf)
        train_ds_eval = _DS(splits_dir / f'{version}_train.txt', data_dir, config["class_to_idx"], transform=eval_tf)
        test_ds       = _DS(splits_dir / f'{version}_test.txt',  data_dir, config["class_to_idx"], transform=eval_tf)

        self.n_train = len(train_ds)
        self.n_test  = len(test_ds)

        pin = (self.device.type == 'cuda')
        nw  = config['num_workers']
        bs  = config['batch_size']

        # Nivel 2: CPU paralela — num_workers processos paralelos pre-processam
        # imagens nos nucleos fisicos enquanto a GPU processa o batch atual
        self.train_loader      = DataLoader(train_ds,      batch_size=bs, shuffle=True,
                                            num_workers=0, pin_memory=pin)
        self.train_eval_loader = DataLoader(train_ds_eval, batch_size=bs, shuffle=False,
                                            num_workers=0, pin_memory=pin)
        self.val_loader        = DataLoader(test_ds,       batch_size=bs, shuffle=False,
                                            num_workers=0, pin_memory=pin)

    def _build_model(self):
        import torch.nn as nn
        from torchvision import models
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, 38)
        return model.to(self.device)

    def train(self):
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from torch.optim.lr_scheduler import StepLR
        from pathlib import Path
        import time

        ver      = self.version
        ckpt_dir = Path(self.config['ckpt_dir'])
        done_path   = ckpt_dir / f'model_{ver}_DONE.pt'
        resume_path = ckpt_dir / f'model_{ver}_resume.pt'
        best_path   = ckpt_dir / f'model_{ver}_best.pt'

        if done_path.exists():
            history = torch.load(done_path, map_location='cpu', weights_only=False)
            print(f'[{ver}] DONE — pulando (melhor val_acc: {max(history["val_acc"]):.4f})')
            return history

        model     = self._build_model()
        optimizer = optim.Adam(model.fc.parameters(), lr=self.config['lr'])
        scheduler = StepLR(optimizer, step_size=3, gamma=0.5)
        criterion = nn.CrossEntropyLoss()
        num_epochs = self.config['num_epochs']

        start_epoch = 0
        history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'best_val_acc': 0.0}

        if resume_path.exists():
            ckpt = torch.load(resume_path, map_location=self.device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            history     = ckpt['history']
            start_epoch = ckpt['epoch'] + 1
            print(f'[{ver}] Retomando do epoch {start_epoch} — device: {self.device}')
        else:
            print(f'[{ver}] Iniciando do zero — device: {self.device}')

        for epoch in range(start_epoch, num_epochs):
            t0 = time.time()

            # Nivel 1: CUDA — GPU processa cada batch inteiro em paralelo
            # nos cores CUDA (operacoes matriciais massivamente paralelas)
            model.train()
            train_loss, train_correct = 0.0, 0
            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                out  = model(imgs)
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()
                train_loss    += loss.item() * imgs.size(0)
                train_correct += (out.argmax(1) == labels).sum().item()

            model.eval()
            val_correct = 0
            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(self.device), labels.to(self.device)
                    val_correct += (model(imgs).argmax(1) == labels).sum().item()

            scheduler.step()

            ep_loss      = train_loss    / self.n_train
            ep_train_acc = train_correct / self.n_train
            ep_val_acc   = val_correct   / self.n_test

            history['train_loss'].append(ep_loss)
            history['train_acc'].append(ep_train_acc)
            history['val_acc'].append(ep_val_acc)

            if ep_val_acc > history['best_val_acc']:
                history['best_val_acc'] = ep_val_acc
                torch.save({'model_state_dict': model.state_dict(),
                            'val_acc': ep_val_acc}, best_path)

            print(f'[{ver}] Epoch {epoch+1}/{num_epochs} | '
                  f'loss={ep_loss:.4f} | train={ep_train_acc:.4f} | '
                  f'val={ep_val_acc:.4f} | {time.time()-t0:.1f}s')

            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history':              history,
            }, resume_path)

        resume_path.unlink(missing_ok=True)
        torch.save(history, done_path)
        print(f'[{ver}] Treino concluido — melhor val_acc: {history["best_val_acc"]:.4f}')
        return history

    def compute_probs(self):
        import torch
        import torch.nn.functional as F
        import numpy as np
        from pathlib import Path

        ver      = self.version
        ckpt_dir = Path(self.config['ckpt_dir'])
        best_path         = ckpt_dir / f'model_{ver}_best.pt'
        train_probs_path  = ckpt_dir / f'train_probs_{ver}.npy'
        test_probs_path   = ckpt_dir / f'test_probs_{ver}.npy'
        train_labels_path = ckpt_dir / f'train_labels_{ver}.npy'
        test_labels_path  = ckpt_dir / f'test_labels_{ver}.npy'

        if all(p.exists() for p in [train_probs_path, test_probs_path,
                                     train_labels_path, test_labels_path]):
            print(f'[{ver}] Probabilidades ja existem — pulando')
            return (
                np.load(train_probs_path), np.load(test_probs_path),
                np.load(train_labels_path), np.load(test_labels_path),
            )

        model = self._build_model()
        ckpt  = torch.load(best_path, map_location=self.device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()

        def _get(loader):
            ps, ls = [], []
            with torch.no_grad():
                for imgs, labels in loader:
                    out = model(imgs.to(self.device))
                    ps.append(F.softmax(out, dim=1).cpu().numpy())
                    ls.append(labels.numpy())
            return np.concatenate(ps), np.concatenate(ls)

        print(f'[{ver}] Calculando probabilidades...')
        train_probs, train_labels = _get(self.train_eval_loader)
        test_probs,  test_labels  = _get(self.val_loader)

        np.save(train_probs_path,  train_probs)
        np.save(test_probs_path,   test_probs)
        np.save(train_labels_path, train_labels)
        np.save(test_labels_path,  test_labels)
        print(f'[{ver}] Probabilidades salvas')
        return train_probs, test_probs, train_labels, test_labels


def build_model_local(device):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 38)
    return model.to(device)


print('ModelTrainer definido')
print(f'  num_gpus por actor : 0.34  (3 actors cabem em 1 GPU | 1 por no no cluster)')
print(f'  num_workers CPU    : {NUM_WORKERS}  (Nivel 2: pre-processamento paralelo)')

ModelTrainer definido
  num_gpus por actor : 0.34  (3 actors cabem em 1 GPU | 1 por no no cluster)
  num_workers CPU    : 0  (Nivel 2: pre-processamento paralelo)


## Treinamento paralelo dos 3 modelos

A celula abaixo lanca os 3 `ModelTrainer` **simultaneamente** via Ray.
Cada actor roda de forma independente no seu no do cluster.

- **Re-executar e sempre seguro:** pula modelos `DONE`, retoma modelos interrompidos.
- **Single node:** 3 actors compartilham a GPU (0.34 cada).
- **Cluster 2 PCs:** Ray aloca 1 actor por GPU automaticamente.


In [ ]:
# === 12. TREINAMENTO PARALELO — Nivel 3 (Ray) ===

# Se todos os modelos ja foram treinados, pula sem criar actors Ray
all_done = all((CKPT_DIR / f'model_{ver}_DONE.pt').exists() for ver in VERSIONS)
if all_done:
    histories = {}
    for ver in VERSIONS:
        h = torch.load(CKPT_DIR / f'model_{ver}_DONE.pt', map_location='cpu', weights_only=False)
        histories[ver] = h
        print(f'[{ver}] DONE — melhor val_acc: {max(h["val_acc"]):.4f}')
    trainers = None
else:
    trainers = {ver: ModelTrainer.remote(ver, CONFIG) for ver in VERSIONS}
    print('Iniciando treino paralelo dos 3 modelos...')
    t_start = time.time()
    futures      = [trainers[ver].train.remote() for ver in VERSIONS]
    results_list = ray.get(futures)
    histories    = dict(zip(VERSIONS, results_list))
    elapsed = time.time() - t_start
    print(f'Treino concluido em {elapsed/60:.1f} min')
    for ver, h in histories.items():
        print(f'  {ver:12s} melhor val_acc: {h["best_val_acc"]:.4f}')


[color] DONE — melhor val_acc: 0.9476
[grayscale] DONE — melhor val_acc: 0.8806
[segmented] DONE — melhor val_acc: 0.9349


In [ ]:
# === 13. CARREGA MELHORES MODELOS ===
best_models = {}
for ver in VERSIONS:
    best_path = CKPT_DIR / f'model_{ver}_best.pt'
    if not best_path.exists():
        print(f'[{ver}] best.pt nao encontrado — execute o treino primeiro')
        continue
    model = build_model_local(DEVICE)
    ckpt  = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    best_models[ver] = model
    print(f'[{ver}] carregado (val_acc: {ckpt["val_acc"]:.4f})')

[color] carregado (val_acc: 0.9476)
[grayscale] carregado (val_acc: 0.8806)
[segmented] carregado (val_acc: 0.9349)


In [ ]:
# === 14. CURVAS DE APRENDIZADO ===
CURVES_PATH = BASE_DIR / 'curvas_aprendizado.png'
if CURVES_PATH.exists():
    print('curvas_aprendizado.png ja existe — pulando')
else:
    # Carrega histories dos DONE.pt caso a celula de treino tenha sido pulada
    if 'histories' not in globals() or histories is None:
        histories = {}
        for ver in VERSIONS:
            done_path = CKPT_DIR / f'model_{ver}_DONE.pt'
            histories[ver] = torch.load(done_path, map_location='cpu', weights_only=False)
        print('histories carregados dos arquivos DONE.pt')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'color': 'tab:green', 'grayscale': 'tab:gray', 'segmented': 'tab:blue'}
    for ver, h in histories.items():
        ep = range(1, len(h['train_loss']) + 1)
        axes[0].plot(ep, h['train_loss'], label=ver, color=colors[ver])
        axes[1].plot(ep, h['val_acc'],    label=ver, color=colors[ver])
    axes[0].set_title('Loss de treino');   axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].set_title('Acc de validacao'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(CURVES_PATH, dpi=120); plt.close()
    print(f'Salvo: {CURVES_PATH}')

curvas_aprendizado.png ja existe — pulando


In [ ]:
# === 15. AVALIACAO INDIVIDUAL DOS 3 MODELOS ===
# Todos os modelos sao avaliados no color_test.txt para comparacao justa com o ensemble.
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

shared_test_ds = PlantVillageDataset(SPLITS_DIR / 'color_test.txt', DATA_DIR, CLASS_TO_IDX, transform=eval_tf)
shared_loader  = DataLoader(shared_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

ind_results = {}
for ver in VERSIONS:
    if ver not in best_models:
        continue
    model = best_models[ver]
    model.eval()
    preds, lbls = [], []
    with torch.no_grad():
        for imgs, labels in shared_loader:
            preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
            lbls.extend(labels.numpy())
    ind_results[ver] = {
        'acc':     accuracy_score(lbls, preds),
        'f1':      f1_score(lbls, preds, average='macro', zero_division=0),
        'prec':    precision_score(lbls, preds, average='macro', zero_division=0),
        'rec':     recall_score(lbls, preds, average='macro', zero_division=0),
        'bal_acc': balanced_accuracy_score(lbls, preds),
    }
    r = ind_results[ver]
    print(f'{ver:12s} acc={r["acc"]:.4f}  f1={r["f1"]:.4f}  prec={r["prec"]:.4f}  rec={r["rec"]:.4f}  bal_acc={r["bal_acc"]:.4f}')

color        acc=0.9476  f1=0.9309  prec=0.9380  rec=0.9266  bal_acc=0.9266
grayscale    acc=0.0751  f1=0.0405  prec=0.1017  rec=0.0594  bal_acc=0.0594
segmented    acc=0.7606  f1=0.6717  prec=0.7950  rec=0.6761  bal_acc=0.6761


In [ ]:
# === 16. HEATMAP DE METRICAS ===
HEATMAP_PATH = BASE_DIR / 'metricas_heatmap.png'
if HEATMAP_PATH.exists():
    print('metricas_heatmap.png ja existe — pulando')
else:
    metrics = ['acc', 'f1', 'prec', 'rec', 'bal_acc']
    rows    = [ver for ver in VERSIONS if ver in ind_results]
    data    = [[ind_results[ver][m] for m in metrics] for ver in rows]
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(data, annot=True, fmt='.4f', xticklabels=metrics,
                yticklabels=rows, cmap='YlGn', vmin=0, vmax=1, ax=ax)
    ax.set_title('Metricas por versao do dataset')
    plt.tight_layout()
    plt.savefig(HEATMAP_PATH, dpi=120); plt.close()
    print(f'Salvo: {HEATMAP_PATH}')

metricas_heatmap.png ja existe — pulando


In [ ]:
# === 17. PROBABILIDADES PARA O ENSEMBLE ===
# Todos os 3 modelos avaliam o MESMO test set (color) para garantir alinhamento.
# As versoes do dataset tem filenames diferentes, entao stacking por linha nao e possivel.
# Ensemble por media de softmax e a abordagem correta.

import torch.nn.functional as F

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Usa o test set de color como conjunto compartilhado
shared_test_ds     = PlantVillageDataset(SPLITS_DIR / 'color_test.txt', DATA_DIR, CLASS_TO_IDX, transform=eval_tf)
shared_test_loader = DataLoader(shared_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

test_probs_all  = {}  # ver -> array (N, 38)
test_labels_all = None

for ver in VERSIONS:
    if ver not in best_models:
        print(f'[{ver}] modelo nao carregado - pulando')
        continue
    model = best_models[ver]
    model.eval()
    ps, ls = [], []
    with torch.no_grad():
        for imgs, labels in shared_test_loader:
            out = model(imgs.to(DEVICE))
            ps.append(F.softmax(out, dim=1).cpu().numpy())
            ls.append(labels.numpy())
    test_probs_all[ver] = np.concatenate(ps)
    if test_labels_all is None:
        test_labels_all = np.concatenate(ls)
    print(f'[{ver}] probs calculadas: {test_probs_all[ver].shape}')

print(f'Labels: {test_labels_all.shape}')

[color] probs calculadas: (10709, 38)
[grayscale] probs calculadas: (10709, 38)
[segmented] probs calculadas: (10709, 38)
Labels: (10709,)


In [ ]:
# === 18. ENSEMBLE POR MEDIA DE SOFTMAX ===
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score

# Media das probabilidades dos 3 modelos
avg_probs     = np.mean([test_probs_all[ver] for ver in VERSIONS if ver in test_probs_all], axis=0)
ensemble_preds = avg_probs.argmax(axis=1)

ens_result = {
    'acc':     accuracy_score(test_labels_all, ensemble_preds),
    'f1':      f1_score(test_labels_all, ensemble_preds, average='macro', zero_division=0),
    'prec':    precision_score(test_labels_all, ensemble_preds, average='macro', zero_division=0),
    'rec':     recall_score(test_labels_all, ensemble_preds, average='macro', zero_division=0),
    'bal_acc': balanced_accuracy_score(test_labels_all, ensemble_preds),
}
r = ens_result
print(f'ensemble      acc={r["acc"]:.4f}  f1={r["f1"]:.4f}  prec={r["prec"]:.4f}  rec={r["rec"]:.4f}  bal_acc={r["bal_acc"]:.4f}')

ensemble      acc=0.9303  f1=0.9085  prec=0.9319  rec=0.8955  bal_acc=0.8955


In [ ]:
# === 19. COMPARACAO FINAL — 3 MODELOS + ENSEMBLE ===
FINAL_PATH = BASE_DIR / 'comparacao_final.png'
if FINAL_PATH.exists():
    print('comparacao_final.png ja existe — pulando')
else:
    all_r = {**ind_results, 'ensemble': ens_result}
    nomes = [v for v in VERSIONS + ['ensemble'] if v in all_r]
    accs  = [all_r[n]['acc'] for n in nomes]
    cores = ['tab:green', 'tab:gray', 'tab:blue', 'tab:orange']
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(nomes, accs, color=cores[:len(nomes)])
    ax.bar_label(bars, fmt='%.4f', padding=3)
    ax.set_ylim(0, 1.05)
    ax.set_title('Acuracia: modelos individuais vs ensemble')
    ax.set_ylabel('Acuracia')
    plt.tight_layout()
    plt.savefig(FINAL_PATH, dpi=120); plt.close()
    print(f'Salvo: {FINAL_PATH}')

Salvo: C:\Users\yluan\Documents\luan\TI6\comparacao_final.png


In [ ]:
# === 20. MATRIZ DE CONFUSAO (ENSEMBLE) ===
CM_PATH = BASE_DIR / 'matriz_confusao_ensemble.png'
if CM_PATH.exists():
    print('matriz_confusao_ensemble.png ja existe — pulando')
else:
    cm = confusion_matrix(test_labels_all, ensemble_preds)
    fig, ax = plt.subplots(figsize=(18, 16))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title('Matriz de Confusao — Ensemble')
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')
    plt.xticks(rotation=90, fontsize=7); plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.savefig(CM_PATH, dpi=120); plt.close()
    print(f'Salvo: {CM_PATH}')

Salvo: C:\Users\yluan\Documents\luan\TI6\matriz_confusao_ensemble.png


In [ ]:
# === 21. RESULTADO FINAL — QUESTAO DE PESQUISA ===
print('=' * 64)
print('QUESTAO DE PESQUISA')
print('=' * 64)
print('O ganho de acuracia do ensemble justifica o custo computacional?')
print()

best_ver = max(ind_results, key=lambda v: ind_results[v]['acc'])
best_ind = ind_results[best_ver]['acc']
ens_acc  = ens_result['acc']
gain     = (ens_acc - best_ind) * 100

print(f'Melhor modelo individual : {best_ver:12s} acc={best_ind:.4f}')
print(f'Ensemble                 :              acc={ens_acc:.4f}')
print(f'Ganho                    : {gain:+.2f} pp')
print()
if gain > 0.5:
    print('Conclusao: SIM — o ensemble supera os modelos individuais de forma relevante.')
elif gain > 0:
    print('Conclusao: MARGINAL — ganho existe mas e pequeno. O custo pode nao se justificar.')
else:
    print('Conclusao: NAO — o ensemble nao supera o melhor modelo individual.')

QUESTAO DE PESQUISA
O ganho de acuracia do ensemble justifica o custo computacional?

Melhor modelo individual : color        acc=0.9476
Ensemble                 :              acc=0.9303
Ganho                    : -1.73 pp

Conclusao: NAO — o ensemble nao supera o melhor modelo individual.
